In [22]:
import tensorflow as tf
import numpy as np
import json
import pandas as pd
import os
from ipywidgets import interact
import ipywidgets as widgets

from Bio import SeqIO
from tqdm import tqdm
from pathlib import Path

from tfr_to_hash import (
    load_tfrecord_to_numpy,
    onehot_to_seq,
    deserialize
)

2025-08-06 11:30:01.949469: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-08-06 11:30:01.955984: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2025-08-06 11:30:02.003865: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2025-08-06 11:30:02.053307: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1754476202.093609 2172539 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1754476202.10

In [23]:
# genome_fasta_files = [ f'data/datasets/ref/human/chr{chromosome}.fa' for chromosome in list(range(1,23)) + ["X", "Y", "MT"]]
# seqs_per_chr = { record.id: str(record.seq).lower() for record in tqdm(SeqIO.parse(genome_fasta, "fasta")) }

In [ ]:
def deserialize(serialized_example, metadata):

    feature_map = {
        'sequence': tf.io.FixedLenFeature([], tf.string),
        'target': tf.io.FixedLenFeature([], tf.string),
    }
    example = tf.io.parse_example(serialized_example, feature_map)
    sequence = tf.io.decode_raw(example['sequence'], tf.bool)
    sequence = tf.reshape(sequence, (metadata['seq_length'], 4))
    sequence = tf.cast(sequence, tf.float32)

    target = tf.io.decode_raw(example['target'], tf.float16)
    target = tf.reshape(target, (metadata['target_length'], metadata['num_targets']))
    target = tf.cast(target, tf.float32)

    return {'sequence': sequence, 'target': target}


def load_tfrecord_to_numpy(tfrecord_path, metadata):
    dataset = tf.data.TFRecordDataset([tfrecord_path], compression_type='ZLIB')
    dataset = dataset.map(lambda x: deserialize(x, metadata))
    sequences = []
    targets = []
    for example in dataset:
        sequences.append(example['sequence'].numpy())
        targets.append(example['target'].numpy())
    sequences = np.stack(sequences)
    targets = np.stack(targets)
    return {'sequence': sequences, 'target': targets}


def onehot_to_seq(arr):
    # arr: (131072, 4), valores float32
    indices = np.argmax(arr, axis=1)
    return ''.join(np.array(['a', 'c', 'g', 't'])[indices])


def get_hash(seq):
    return hashlib.sha256(seq.encode()).hexdigest()


def find_matches(genome_fasta, tfrecord_hashes, window_size=131072):
    for record in SeqIO.parse(genome_fasta, "fasta"):
        chrom = record.id
        seq = str(record.seq).upper()
        matches = []
        for i in range(len(seq) - window_size + 1):
            subseq = seq[i:i+window_size]
            h = get_hash(subseq)
            if h in tfrecord_hashes:
                matches.append((chrom, i, i+window_size, h))
                print(f"Match at {chrom}:{i}-{i+window_size}")
        return matches  # podés guardar o devolver esto


def get_all_hashes():
    hashes = [ open("sequence_hashes/"+x, "rt").readlines() for x in os.listdir("sequence_hashes") ]
    return hashes


def flatten_list(lst):
    return[x for y in lst for x in y]


def write_fasta(sequences, output_path):
    with open(output_path, "w") as f:
        for name, seq in sequences:
            f.write(f">{name}\n")
            # cortar en líneas de 80 caracteres
            for i in range(0, len(seq), 80):
                f.write(seq[i:i+80] + "\n")


def get_coords_from_blast(record_id):
    return pd.read_csv(f"alignments/{record_id}_blast_results_subset_top.tsv", sep='\t', header=None).set_axis(["seqID", "chromosome", "start", "end", "e_value", "bitscore"], axis=1)


def extract_region(region, expected_length=131_072):
    if not str(region.chromosome).startswith("chr"):
        chromosome = f"chr{region.chromosome}"
    else:
        chromosome = region.chromosome
    start = int(region.start)
    end   = int(region.end)
    sequence = mm10_per_chr[chromosome][start:end]

    assert (len(sequence)) == expected_length, f"Difference between start and end is not {expected_length} but {len(sequence)}."
    return sequence


def get_all_mouse_sequences(path="data/datasets/basenji/mouse/tfrecords/"):
    
    metadata = {
      'seq_length': 131072,
      'target_length': 896,
      'num_targets': 1643
    }
    all_seqs = []
    for basename in tqdm(os.listdir(path)):
        tfrecord_path = path + basename
        record_data = load_tfrecord_to_numpy(tfrecord_path, metadata)
        seqs = record_data['sequence'].astype(np.int8)
        all_seqs.append(seqs)
    
    all_seqs = np.concatenate(all_seqs, axis=0)
    return all_seqs


def get_hg38():
    genome_fasta = f"{HUMAN_REF_FOLDER}/hg38_chr1_to_X.fa"
    seqs_per_chr = { record.id: str(record.seq).lower() for record in tqdm(SeqIO.parse(genome_fasta, "fasta")) }
    return seqs_per_chr


def get_mm10():
    genome_fasta = f"{MOUSE_REF_FOLDER}/mm10.fa"
    seqs_per_chr = { record.id: str(record.seq).lower() for record in tqdm(SeqIO.parse(genome_fasta, "fasta")) }
    return seqs_per_chr
    

def get_human_record_ids():
        
    remove_extension = lambda filename: filename.replace(".tfr", "")
    return sorted(
        [ remove_extension(f) for f in os.listdir(HUMAN_TFR_FOLDER) ], 
        key=lambda x: (x.split('-')[0], int(x.split('-')[2]))
    )


def get_mouse_record_ids():
    
    return sorted(
        [ f.replace(".tfr", "") for f in os.listdir(MOUSE_TFR_FOLDER) ], key=lambda x: (x.split('-')[0], int(x.split('-')[2]))
    )


def get_sequences_for_record(record_id):
    seq_for_record = { 
        record.id: str(record.seq).lower() 
        for record in tqdm(SeqIO.parse(f"fasta_seq/{record_id}.fasta", "fasta"))
    }    
    return seq_for_record


def get_human_basenji_regions():
    human_seqs = pd.read_csv("data/datasets/basenji/human/sequences.bed", sep='\t', header=None)
    human_seqs = human_seqs.set_axis(["chromosome", "start", "end", "subset"], axis=1)
    return human_seqs


def get_mouse_basenji_regions():
    mouse_seqs = pd.read_csv("data/datasets/basenji/mouse/sequences.bed", sep='\t', header=None)
    mouse_seqs = mouse_seqs.set_axis(["chromosome", "start", "end", "subset"], axis=1)
    return mouse_seqs


def expand_regions(regions_df, to_left:int=131_072, to_right:int=131_072):
    regions_df.start -= to_left
    regions_df.end += to_right
    return regions_df


def get_region_from_npy_filename(filename):
    return {
      'chr': filename.split("_")[0], 
      'start': int(filename.split("_")[1]), 
      'end': int(filename.split("_")[2].split(".")[0])
    }

In [ ]:
MOUSE_REF_FOLDER = "data/datasets/ref/mouse"
HUMAN_REF_FOLDER = "data/datasets/ref/human"
HUMAN_TFR_FOLDER = "data/datasets/basenji/human/tfrecords"
MOUSE_TFR_FOLDER = "data/datasets/basenji/mouse/tfrecords"

mm10_per_chr = get_mm10()
mouse_seqs_df = get_mouse_basenji_regions()
mouse_seqs_set = set(mouse_seqs_df.apply(extract_region, axis=1).to_list())

66it [00:08,  7.51it/s]


In [ ]:
train_files = [ f"{MOUSE_TFR_FOLDER}/train-1-{i}.tfr" for i in range(200) ]
valid_files = [ f"{MOUSE_TFR_FOLDER}/valid-1-{i}.tfr" for i in range(200) ]
test_files = [ f"{MOUSE_TFR_FOLDER}/test-1-{i}.tfr" for i in range(200) ]
train_files = [ f for f in train_files if os.path.exists(f) ]
valid_files = [ f for f in valid_files if os.path.exists(f) ]
test_files = [ f for f in test_files if os.path.exists(f) ]

train_files = train_files + valid_files

In [ ]:
metadata = dict(seq_length=131072, target_length=896, num_targets=1643)
records = load_tfrecord_to_numpy(test_files[0], metadata=metadata)

2025-08-06 11:31:33.880108: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: CUDA_ERROR_NO_DEVICE: no CUDA-capable device is detected
2025-08-06 11:31:33.994567: E tensorflow/core/util/util.cc:131] oneDNN supports DT_HALF only on platforms with AVX-512. Falling back to the default Eigen-based implementation if present.
2025-08-06 11:31:34.003295: I tensorflow/core/kernels/data/tf_record_dataset_op.cc:381] TFRecordDataset `buffer_size` is unspecified, default to 262144
2025-08-06 11:31:37.112700: I tensorflow/core/framework/local_rendezvous.cc:407] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


In [ ]:
MOUSE_TFR_FOLDER = "./datasets/basenji/mouse/tfrecords"
train_files = [ f"{MOUSE_TFR_FOLDER}/train-1-{i}.tfr" for i in range(200) ]
valid_files = [ f"{MOUSE_TFR_FOLDER}/valid-1-{i}.tfr" for i in range(200) ]
test_files = [ f"{MOUSE_TFR_FOLDER}/test-1-{i}.tfr" for i in range(200) ]
train_files = [ f for f in train_files if os.path.exists(f) ]
valid_files = [ f for f in valid_files if os.path.exists(f) ]
test_files = [ f for f in test_files if os.path.exists(f) ]

In [ ]:
import h5py

with h5py.File("test_mouse_copy.h5", "r+") as f:
    dset_target = f["target"]
    num_samples = dset_target.shape[0]

    start = 0
    for i, test_file in enumerate(test_files):
    # for start in tqdm(range(0, num_samples, chunk_size), desc="Escribiendo targets"):
        # end = min(start + chunk_size, num_samples)                
        tgt_chunk = load_tfrecord_to_numpy(test_file, metadata=metadata)['target']
        end = start + len(tgt_chunk)
        print(i, start, end)
        dset_target[start:end] = tgt_chunk
        start = end

In [38]:
with h5py.File("test_mouse_copy.h5", "r+") as f:
    dset_target = f["target"]
    num_samples = dset_target.shape[0]

    start = 0
    for i, test_file in enumerate(test_files):
    # for start in tqdm(range(0, num_samples, chunk_size), desc="Escribiendo targets"):
        # end = min(start + chunk_size, num_samples)                
        tgt_chunk = load_tfrecord_to_numpy(test_file, metadata=metadata)['target']
        end = start + len(tgt_chunk)
        print(i, start, end)
        dset_target[start:end] = tgt_chunk
        start = end

0 0 256


2025-08-06 11:36:48.965203: I tensorflow/core/framework/local_rendezvous.cc:407] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


1 256 512
2 512 768
3 768 1024
4 1024 1280


2025-08-06 11:37:10.143039: I tensorflow/core/framework/local_rendezvous.cc:407] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


5 1280 1536
6 1536 1761


In [32]:
records['target'].shape

(256, 896, 1643)

In [93]:
def chunk_by_subset(df, chunk_size=256):
    result = {}

    for subset in ["train", "valid", "test"]:
        df_subset = df[df["subset"] == subset]
        chunks = [df_subset.iloc[i:i+chunk_size] for i in range(0, len(df_subset), chunk_size)]
        for i, chunk in enumerate(chunks):
            name = f"{subset}-1-{i}"
            result[name] = chunk

    return result


In [ ]:
for i, (record_id, region_df) in enumerate(file_to_chunk.items()):
    # print(record_id)
    hashes_for_extracted = [ hash(x) for x in region_df.apply(extract_region, axis=1).to_list() ]
    sequences_for_record = [ v for k, v in get_sequences_for_record(record_id).items() ]
    hashes_for_fasta = [ hash(x) for x in sequences_for_record ]
    print(all([ hashes_for_extracted[i] == hashes_for_fasta[i] for i in range(256)]))

256it [00:00, 3493.30it/s]


True


0it [00:00, ?it/s]

256it [00:00, 3826.01it/s]


True


...

256it [00:00, 3897.03it/s]


True


256it [00:00, 4269.91it/s]


True


0it [00:00, ?it/s]

In [77]:
file_to_chunk['valid-1-8'].apply(extract_region, axis=1)

31343    tataagttcactattttaaaattatcttccgctatgccccagcaag...
31344    tccctcatgtatcctggacctctggctcctgttaccgcccccacag...
31345    cagttattatgaccttgggctcacctcctaactcctagagactccc...
31346    aatttactctccagaaaagcctaatttgaactctgtccacttacaa...
31347    agatgctaataacatcacgagagattaaattttaaagcaaaataca...
                               ...                        
31499    aacaagcatacggtggcatttcctgacaaagaagcttagcatcctc...
31500    aagacacaggaatcagaaatcccacttgttcacacaatcagatgtc...
31501    ttagttcatattgttgttcctcctgtgggcctgcaaacccattcag...
31502    aacaccacctacagaggagtctagcaaaagtgctgtgttctgttat...
31503    taagttgacatgctaataggaaataattgatgaaagatttcacaga...
Length: 161, dtype: object

In [ ]:
RECORD_ID = 'train-1-0'
sequences_from_tfr = get_sequences_for_record(RECORD_ID)
file_to_chunk = chunk_by_subset(mouse_seqs_df)
mouse_seqs_df.query("subset == 'train'").apply(extract_region, axis=1)

sequences_from_tfr

256it [00:00, 2953.97it/s]


TypeError: chunk_and_map_filenames() missing 1 required positional argument: 'df'

0        tattttggatattccctctatcaaatgcagagttagtgaagatctt...
1        gattccaagaggaagtgaatggcacaaatgtgagataataaaaaat...
2        gataatttccaggaaggtgagcaggaggctcagcaggcacagcaga...
3        gaagcaaaaacatgggacatttcttactgccttgctctccctagtt...
4        ctgttagatctcaactaacacggatgacagcatgtacctccttgtc...
                               ...                        
29290    ctaaatttcatgtgtggtaatttctgtatatgtgtctctaccttaa...
29291    tagtatccatgcccatcatggtacagggcaagcagtaggcaggcag...
29292    attcacacatttacacatgtgcccctacaagaacacacacatatag...
29293    taacaaaaacatttaattagtttataaaattatttaaatttcttaa...
29294    tatggaggcccaaagttgatccccaatcattcttccatcttattta...
Length: 29295, dtype: object

In [24]:
SEQLEN = 131072
TO_LEFT, TO_RIGHT = SEQLEN, SEQLEN
WHICH_RECORD = 5

record_id = (record_ids := get_mouse_record_ids())[WHICH_RECORD]

print(f"{record_id=}")
sequences_from_tfr = get_sequences_for_record(record_id)
blast_results_df   = get_coords_from_blast(record_id)

expanded_regions_df = expand_regions(blast_results_df, to_left=1+TO_LEFT, to_right=0+TO_RIGHT)
sequences_from_ref  = expanded_regions_df.apply(extract_region, axis=1, expected_length=SEQLEN+TO_LEFT+TO_RIGHT)

# if this gives True, we are good
assert all([ v == sequences_from_ref.iloc[i][TO_LEFT:-TO_RIGHT] for i, (k, v) in enumerate(sequences_from_tfr.items()) ])

record_id='test-1-6'


256it [00:00, 2863.95it/s]


In [85]:
subset = ['train', 'valid', 'test'][2]
mouse_seqs.query("subset == @subset")

,chromosome,start,end,subset
31504,chr3,82133605,82264677,test
31505,chrX,61204799,61335871,test
31506,chr12,32669497,32800569,test
31507,chr12,104156470,104287542,test
31508,chr2,86622742,86753814,test
...,...,...,...,...
33516,chr2,110707432,110838504,test
33517,chr12,59936948,60068020,test
33518,chrX,56644074,56775146,test
33519,chrX,23595900,23726972,test


In [ ]:
region = region_from_npy_filename("chr1_100033626_100426841.npy")
npy_file = (npy_files := os.listdir(MOUSE_NPY_FOLDER))[0]
one_sequence_from_before = onehot_to_seq(np.load(f"{MOUSE_NPY_FOLDER}/{npy_file}"))

In [113]:
SHIFT = -3
one_sequence_from_before[SEQLEN+SHIFT:-SEQLEN+SHIFT] in mouse_seqs_set

False

In [ ]:
sequences.iloc[1] in mouse_seqs_set

True

In [18]:
@interact
def show_sequence(start_index=widgets.IntSlider(min=1000000, max=60000000, step=100000)):
    print(chromosome_seq[start_index:(start_index+1000)])